# LLM Projesi Final Aşaması - Kaggle Çalıştırıcısı

Bu notebook, bilgisayarınızdaki `.py` dosyalarını Kaggle üzerinde sırasıyla çalıştırmanızı sağlar.

**ÖNEMLİ BİLGİ:** Kaggle'a verilerinizi (`countries.json`, `eval_set.json`) ve Python dosyalarınızı bir Dataset (Add Input) olarak eklediğinizden emin olun.
Kaggle Dataset'leri salt okunur (read-only) olduğu için, eğer `.py` dosyalarınızı direkt datasetten çalıştırırsanız hata alabilirsiniz. Bu nedenle kod hücreleri sizin yerinize bu dosyaları `/kaggle/working` dizinine kopyalayacaktır.

In [ ]:
# Gerekli kütüphaneleri kuruyoruz
!pip install -q "transformers==4.44.2" "trl==0.11.4" "peft==0.13.2" \
             "accelerate==0.34.2" "datasets==2.21.0" "bitsandbytes==0.43.3" \
             sentence-transformers chromadb langchain_text_splitters \
             faiss-cpu seaborn matplotlib pandas tqdm psutil google-generativeai

In [ ]:
# EĞER PYTHON DOSYALARINIZI KAGGLE'A DATASET OLARAK YÜKLEDİYSENİZ:
# Dataset yolunuzu buraya girin ve scriptleri working klasörüne kopyalayın.
# Dataset adınız örneğin 'cfbook' ise:
# !cp -r /kaggle/input/cfbook/* /kaggle/working/
# !chmod +x /kaggle/working/*.py

## 1. Adım: RAG Pipeline (ZORUNLU İLK ADIM)
Bu dosya vektör veritabanını ve `pipeline_config.json` dosyasını oluşturur. Bu adım tamamlanmadan diğer hiçbir kod ÇALIŞMAZ.

In [ ]:
!python Final_RAG_Pipeline.py

## 2. Adım: Fine-Tuning (Modelleri Eğitme)

**NEDEN AYRI AYRI ÇALIŞTIRMALISINIZ?**
Çünkü her bir Büyük Dil Modelini (LLM) fine-tune etmek GPU'nun belleğini (VRAM) sonuna kadar doldurur. Modelleri arka arkaya aynı oturumda (session) çalıştırırsanız **CUDA Out of Memory (OOM)** hatası alırsınız.

**Çözüm:**
Önce Qwen'i çalıştırın. Bittiğinde Kaggle'ın üst menüsünden **Run -> Restart Session** diyerek GPU hafızasını boşaltın. Sonra diğer modele (Llama3 veya Phi35) geçin.

In [ ]:
!python Final_QLoRA_Finetune_Qwen.py

In [ ]:
# Önceki model bittikten sonra Restart Session yapıp bu hücreyi aktifleştirebilirsiniz.
# !python Final_QLoRA_Finetune_Llama3.py

In [ ]:
# Llama3 bittikten sonra Restart Session yapıp bu hücreyi aktifleştirebilirsiniz.
# !python Final_QLoRA_Finetune_Phi35.py

## 3. Adım: Değerlendirme (Evaluator)
Eğittiğiniz veya test edeceğiniz her model için `Final_Model_Evaluator.py` dosyasını çalıştırmalısınız.

In [ ]:
# Qwen 3B modeli test ediliyor
!sed -i 's/MODEL_KEY = .*/MODEL_KEY = "qwen_3b"/g' Final_Model_Evaluator.py
!python Final_Model_Evaluator.py

In [ ]:
# Llama3 modeli test edilecekse (Kaggle Secrets'ta HF_TOKEN kayıtlı olmalı):
# !sed -i 's/MODEL_KEY = .*/MODEL_KEY = "llama3_8b"/g' Final_Model_Evaluator.py
# !python Final_Model_Evaluator.py

In [ ]:
# Phi 3.5 modeli test edilecekse:
# !sed -i 's/MODEL_KEY = .*/MODEL_KEY = "phi_35"/g' Final_Model_Evaluator.py
# !python Final_Model_Evaluator.py

## 4. Adım: Gemini API (Opsiyonel) ve Son Karşılaştırma
Gemini testi yapmak isterseniz Final_Gemini_API_Test.py'yi çalıştırın. En son `Final_Comparison.py` çalıştırılarak nihai grafikler ve Markdown raporu oluşturulur.

In [ ]:
# Opsiyonel Gemini Testi (Kaggle Secrets'ta GEMINI_API_KEY tanımlanmış olmalıdır):
# !python Final_Gemini_API_Test.py

In [ ]:
# ZORUNLU: Ana karşılaştırma tablosunu oluşturur
!python Final_Comparison.py

## 5. Adım: Bonus Uygulamalar (Ek Puan)
Hocadan ek puan (veya projeyi derinleştirmek) için yazılmış, ana akıştan bağımsız özel dosyalar. İstediğiniz zaman çalıştırabilirsiniz.

In [ ]:
# BONUS 1: FAISS vs ChromaDB Karşılaştırması
# RAG Pipeline'a bağımlıdır (pipeline_config.json'a ihtiyaç duyar).
!python Final_FAISS_Retriever.py

In [ ]:
# BONUS 2: Sentetik Eğitim Verisi Üretimi (Gemini Kullanarak)
# Kaggle Secrets'ta GEMINI_API_KEY varsa API atar, yoksa demo modda şablon üretir.
!python Final_Synthetic_Data_Generator.py

In [ ]:
# BONUS 3: LoRA Konfigürasyon Karşılaştırması
# Qwen modelinde farklı LoRA r ve alpha değerlerini (r=8, 16, 32 vb.) karşılaştırır. 
# DİKKAT: RAM/GPU doldurabilir. Öncesinde 'Restart Session' yapmanız tavsiye edilir.
!python Final_LoRA_Config_Comparison.py